In [2]:
import os
import sys
from pathlib import Path
from typing import Dict, Optional, Tuple
from langchain_core.messages import HumanMessage, SystemMessage

PROJECT_ROOT = os.getcwd()
SKILLS_DIR   = os.path.join(PROJECT_ROOT , "../skills")
AGENT_FILE   = os.path.join(PROJECT_ROOT , "../agent/skill_agent.py")

print(f"Project root: {PROJECT_ROOT}")
print(f"Skills directory: {SKILLS_DIR}")
print(f"Agent file: {AGENT_FILE}")

Project root: /mnt/e/Personal/Samarth/repository/skills_with_langchain/Dynamic-Skill-Creation/src/cookbooks
Skills directory: /mnt/e/Personal/Samarth/repository/skills_with_langchain/Dynamic-Skill-Creation/src/cookbooks/../skills
Agent file: /mnt/e/Personal/Samarth/repository/skills_with_langchain/Dynamic-Skill-Creation/src/cookbooks/../agent/skill_agent.py


In [3]:
sys.path.append(os.path.dirname(PROJECT_ROOT))
from handler.skills_registry import load_skill_registry

def test_registry():
    print("=" * 50)
    print("TEST 1: Skill Registry Loading")
    print("=" * 50)
    registry = load_skill_registry()
    assert len(registry) > 0, "Registry should not be empty"
    print(f"✓ Loaded {len(registry)} skills: {list(registry.keys())}")

    return registry

In [4]:
import os
import sys
from pathlib import Path
from typing import Dict, Optional, Tuple
from langchain_core.messages import HumanMessage, SystemMessage
sys.path.append(os.path.dirname(PROJECT_ROOT))
from utils.content_reader import fetch_url_content, read_markdown_file

def read_markdown_file(uploaded_file) -> Tuple[str, str]:
    """
    Read content from a Streamlit UploadedFile object (.md / .txt / .rst).
 
    Args:
        uploaded_file : st.file_uploader result (must not be None).
 
    Returns:
        (content, source_hint)
        content     — full UTF-8 text of the file
        source_hint — human-readable label, e.g. "Uploaded file: SKILL.md"
 
    Raises:
        ValueError  — if the file cannot be decoded as UTF-8
        RuntimeError — if uploaded_file is None
    """
    if uploaded_file is None:
        raise RuntimeError("No file provided — uploaded_file is None.")
 
    try:
        raw_bytes = uploaded_file.read()
        content   = raw_bytes.decode("utf-8")
    except UnicodeDecodeError:
        # Try latin-1 as a fallback before giving up
        try:
            content = raw_bytes.decode("latin-1")
            
        except Exception as exc:
            raise ValueError(
                f"Cannot decode '{uploaded_file.name}' as UTF-8 or latin-1: {exc}"
            ) from exc
 
    source_hint = f"Uploaded file: {uploaded_file.name}"
    return content.strip(), source_hint

### Using Builtin Scraper

In [6]:
import os
import sys
from pathlib import Path
from typing import Dict, Optional, Tuple
from langchain_core.messages import HumanMessage, SystemMessage
sys.path.append(os.path.dirname(PROJECT_ROOT))
from utils.content_reader import fetch_url_content, read_markdown_file
import re 
file_path = "/mnt/e/Personal/Samarth/repository/skills_with_langchain/Implementing_skills_using_langchain/README.md"
url_path = "https://www.google.com/search?q=who+is+the+current+president+of+iran"
content = fetch_url_content(url_path)
print(content)

Google Search
Please click 
here
 if you are not redirected within a few seconds.
If you're having trouble accessing Google Search, please 
click here
, or send 
feedback
.


### Using FireCrawl

In [ ]:
from langchain_community.document_loaders.firecrawl import FireCrawlLoader
loader = FireCrawlLoader(
    api_key="fc-", 
    url="https://github.com/langchain-ai/deepagents/blob/main/README.md",
    mode="scrape"
)
data = loader.load()

#### parsing the jupyter notebook

In [ ]:
from jupyter_notebook_parser import JupyterNotebookParser
### Local notebook parsing
parsed = JupyterNotebookParser('sam3_image_predictor_example.ipynb')

# Get all code cell sources as a list of strings
code_sources = parsed.get_code_cell_sources()

# Iterate and print the raw source of each code cell
for source in code_sources:
    print(source.raw_source)

In [ ]:
import requests
import nbformat

def parse_notebook_from_url(url):
    # 1. Fetch the raw content
    response = requests.get(url)
    if response.status_code == 200:
        notebook_content = response.text
        # 2. Parse the Jupyter Notebook content
        nb = nbformat.reads(notebook_content, as_version=4)
        return nb
    else:
        print("Failed to download notebook")
        return None

url = "https://github.com/facebookresearch/sam3/blob/main/examples/sam3_image_predictor_example.ipynb"
nb = parse_notebook_from_url(url)

# Iterate through cells
if nb:
    for cell in nb.cells:
        print(f"Cell Type: {cell.cell_type}")
        print(f"Source:\n{cell.source}\n{'-'*20}")

#### Testing the Internet search 

In [19]:
import os
import sys
import re 
from pathlib import Path
from typing import Dict, Optional, Tuple
from langchain_core.messages import HumanMessage, SystemMessage
sys.path.append('/mnt/e/Personal/Samarth/repository/skills_with_langchain/Dynamic-Skill-Creation/src/skills/browser-search/scripts')
from browser_search import run_browser_search


run_browser_search("What is the capital of France ?")

SerpAPI search failed for query 'What is the capital of France ?': {'status': 'error', 'url': 'What is the capital of France ?', 'confidence': 'low', 'source': 'serpapi', 'metadata': {'error': '1 validation error for SerpAPIWrapper\n  Value error, Did not find serpapi_api_key, please add an environment variable `SERPAPI_API_KEY` which contains it, or pass `serpapi_api_key` as a named parameter. [type=value_error, input_value={}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.12/v/value_error'}}


{'status': 'success',
 'url': 'Paris is thecapitalofFrance(French Republic), situated in the Western Europe subregion of Europe. In Paris, the currency used is Euro (€), which is the official currency used inFrance. Paris, thecapitaland largest city ofFrance,issituated in the north-central region of the country along the Seine River.) The city has a rich history, with evidence of human habitation dating back to approximately 7600 BCE.) Over the centuries, Paris has evolved into a global center for art, fashion, and culture, attracting millions of visitors annually.) WhatmakesFrancea unique country to travel to? Country DescriptionFranceisa developed and stable democracy with a modern economy. Tourist facilities are widely available. CrimeFranceisa relatively safe country. Most crimes are non-violent, but pick-pocketing is a significant problem. The majority of crimes directed against foreign visitors, including U.S. citizens, involve pick ... Capitalcities of Europe in alphabetical ord

In [ ]:
from langchain_community.utilities import GoogleSerperAPIWrapper
import os
import pprint

os.environ["SERPER_API_KEY"] = "YOUR_SERPER_API_KEY"
search = GoogleSerperAPIWrapper()

In [5]:
search.run("Who was the previous supreme leader of Iran  ?")

"The Islamic Republic of Iran has in its history had three supreme leaders: Khomeini, who held the position from 1979 until his death in 1989; Ali Khamenei, who ... Ruhollah Mostafavi Musavi Khomeini (17 May 1900 – 3 June 1989) was an Iranian political revolutionary and Shia cleric who served as the first supreme leader ... This comprehensive biography of Ali Khamenei explores the life, power, and controversial legacy of the Islamic Republic of Iran's Supreme ... Mojtaba Khamenei was well postured within the regime to take the reins: he effectively ran the Office of the Supreme Leader for decades and worked closely with ... Ayatollah Ali Khamenei: Former Supreme Leader of Iran · Mojtaba Khamenei: The Third Supreme Leader of Iran · Brigadier General Esmail Qaani: Commander of the ... As one of Iran's longest-serving leaders, Khamenei was almost as ubiquitous in Iranian society as his predecessor, Ayatollah Ruhollah Khomeini, ... At the top of Iran's power structure is the Supreme Leader

In [ ]:
from langchain_core.messages import HumanMessage, SystemMessage, ToolMessage, AIMessage
response = {'messages': [HumanMessage(content='what is the capital of france ', additional_kwargs={}, response_metadata={}, id='ce9c4882-90bd-4be8-8f01-d8f0e9ef47be'), AIMessage(content='To answer your question about the capital of France, I can provide you with the information directly without needing to use any tools.\n\nThe capital of France is Paris.', additional_kwargs={}, response_metadata={'model': 'qwen2.5:latest', 'created_at': '2026-04-04T20:20:41.097381555Z', 'done': True, 'done_reason': 'stop', 'total_duration': 53765892211, 'load_duration': 94254266, 'prompt_eval_count': 4096, 'prompt_eval_duration': 49297195834, 'eval_count': 33, 'eval_duration': 4219325853, 'logprobs': None, 'model_name': 'qwen2.5:latest', 'model_provider': 'ollama'}, name='DynamicSkillAgent', id='lc_run--019d5a26-d400-7330-81d5-6f8c409b7b5c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 4096, 'output_tokens': 33, 'total_tokens': 4129})]}
print(response['messages'][1].content)

To answer your question about the capital of France, I can provide you with the information directly without needing to use any tools.

The capital of France is Paris.


In [66]:
response = {'messages': [HumanMessage(content='look up , who is new supreme leader of Iran ', additional_kwargs={}, response_metadata={}, id='803006e9-05a1-4472-a203-6e63c880993e'), AIMessage(content='', additional_kwargs={}, response_metadata={'model': 'qwen2.5:latest', 'created_at': '2026-04-05T05:28:33.547007734Z', 'done': True, 'done_reason': 'stop', 'total_duration': 52175763667, 'load_duration': 90583194, 'prompt_eval_count': 4096, 'prompt_eval_duration': 49179518716, 'eval_count': 25, 'eval_duration': 2780671460, 'logprobs': None, 'model_name': 'qwen2.5:latest', 'model_provider': 'ollama'}, name='DynamicSkillAgent', id='lc_run--019d5c1c-7239-7370-8807-2c3fa99eff99-0', tool_calls=[{'name': 'browser_search_tool', 'args': {'query': 'new supreme leader of Iran'}, 'id': '520e763c-16de-4ce4-b050-2a89fb4e1990', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 4096, 'output_tokens': 25, 'total_tokens': 4121}), ToolMessage(content='{\n    "status": "success",\n    "url": "In a post on X written in Farsi, the Israeli military declared it would continue \\"pursuing every successorofIran\'s latesupremeleader\\" — and go ... Anewstatement was issued in the nameofIran’sSupremeLeaderAyatollah Mojtaba Khamenei on Monday, amid persistent questions about his ... However, authorities have not yet revealed the identityofthenewSupremeLeaderor announced when the official announcement will be made. People hold placards with imagesofIran’snewsupremeleaderMojtaba Khamenei during a gathering to support him, amid the U.S.-Israeli conflict ... ... official long considered oneofthe most influential but least visible figures inIran’s political establishment has been named the country’snew..."\n}', name='browser_search_tool', id='65ccbb9b-ff79-4a48-89d9-3889035ea874', tool_call_id='520e763c-16de-4ce4-b050-2a89fb4e1990'), AIMessage(content="Based on the provided text, it appears that there is information about Iran's new Supreme Leader. Let me summarize and extract the key points:\n\n- A new statement was issued in the name of Iran’s Supreme Leader Ayatollah Mojtaba Khamenei.\n- This announcement came amid persistent questions about his identity.\n- The identity of the new Supreme Leader has not been officially revealed yet.\n- There are speculations that this could be one of the most influential but least visible figures in Iran's political establishment.\n\nWould you like me to perform any specific action based on this information, or do you have a particular question related to it?", additional_kwargs={}, response_metadata={'model': 'qwen2.5:latest', 'created_at': '2026-04-05T05:29:41.606368696Z', 'done': True, 'done_reason': 'stop', 'total_duration': 66040791195, 'load_duration': 101741599, 'prompt_eval_count': 4096, 'prompt_eval_duration': 49214848473, 'eval_count': 129, 'eval_duration': 16348487030, 'logprobs': None, 'model_name': 'qwen2.5:latest', 'model_provider': 'ollama'}, name='DynamicSkillAgent', id='lc_run--019d5c1d-45ec-7830-8c87-7844e7b4935e-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 4096, 'output_tokens': 129, 'total_tokens': 4225})]}
AgentState = {
        "messages": [HumanMessage(content='look up , who is new supreme leader of Iran')],
        "selected_skill": None,
        "skill_instructions": None,
        "tool_results": [],
        "final_response": None,
        "token_usage": {"input_tokens": 0, "output_tokens": 0, "total_tokens": 0}
    }
for msg in response['messages']:
        # --- HUMAN MESSAGE ---
        if isinstance(msg, HumanMessage):
            # already added manually, skip or append if needed
            continue

        # --- AI MESSAGE ---
        elif isinstance(msg, AIMessage):
            # accumulate token usage
            usage = getattr(msg, "usage_metadata", {})
            if usage:
                AgentState["token_usage"]["input_tokens"] += usage.get("input_tokens", 0)
                AgentState["token_usage"]["output_tokens"] += usage.get("output_tokens", 0)
                AgentState["token_usage"]["total_tokens"] += usage.get("total_tokens", 0)

            # if AI made tool calls → intermediate step
            if msg.tool_calls:
                AgentState["selected_skill"] = msg.tool_calls[0].get("name")

            # if AI has actual content → final answer
            if msg.content:
                AgentState["final_response"] = msg.content

        # --- TOOL MESSAGE ---
        elif isinstance(msg, ToolMessage):
            AgentState["tool_results"].append({
                "tool_name": msg.name,
                "content": msg.content,
                "tool_call_id": msg.tool_call_id
            })


# tools_calls = []
# for msg in response['messages']:
#     if (msg.__class__.__name__) == 'AIMessage':
#         if len(msg.tool_calls)>0:
#             tools_calls.append((msg.tool_calls[0]['name']))
#         print(msg)

            

        #print(f"{msg.__class__.__name__}: {msg.content}\n{'-'*50}")

    #print("Tool Calls: {msg.response_metadata['tool_calls']}\n{'-'*50}")
    #print(f"{msg.__class__.__name__}: {msg.content}\n{'-'*50}")


In [67]:
AgentState

{'messages': [HumanMessage(content='look up , who is new supreme leader of Iran', additional_kwargs={}, response_metadata={})],
 'selected_skill': 'browser_search_tool',
 'skill_instructions': None,
 'tool_results': [{'tool_name': 'browser_search_tool',
   'content': '{\n    "status": "success",\n    "url": "In a post on X written in Farsi, the Israeli military declared it would continue \\"pursuing every successorofIran\'s latesupremeleader\\" — and go ... Anewstatement was issued in the nameofIran’sSupremeLeaderAyatollah Mojtaba Khamenei on Monday, amid persistent questions about his ... However, authorities have not yet revealed the identityofthenewSupremeLeaderor announced when the official announcement will be made. People hold placards with imagesofIran’snewsupremeleaderMojtaba Khamenei during a gathering to support him, amid the U.S.-Israeli conflict ... ... official long considered oneofthe most influential but least visible figures inIran’s political establishment has been nam